# JPAIS Sovereign Legal AI - Interactive Exploration
This notebook interacts directly with your local bare-metal stack:
- **Neo4j:** Relational and hierarchical regulatory graph
- **Qdrant:** 768-dimensional legal vector collections
- **Ollama (Qwen 2.5 3B):** Local GPU inference on Quadro T2000

In [1]:
# 1. Query Neo4j Knowledge Graph
from neo4j import GraphDatabase
driver = GraphDatabase.driver("bolt://127.0.0.1:7687", auth=("neo4j", "JpaisLegalGraph2026!"))
with driver.session() as session:
    result = session.run("""
        MATCH (r:Regulation)-[:CONTAINS]->(a:Article)-[:REFERENCES]->(lc:LegalConcept)
        RETURN r.title AS reg_title, a.article_number AS art_num, a.content AS content, lc.canonical_name AS concept
    """)
    for row in result:
        print(f"=== {row.get('reg_title')} ===")
        print(f"{row.get('art_num')} (Konsep: {row.get('concept')}):")
        print(f"  {row.get('content')}")
driver.close()

=== Undang-Undang Tata Kelola Kecerdasan Artifisial Berdaulat ===
Pasal 12 (Konsep: Pencatatan Asal-Usul Data (Provenance Logging)):
  Pasal 12 ayat (1): Setiap institusi peradilan yang mengoperasikan kecerdasan buatan wajib mempertahankan log asal-usul data (provenance) secara kriptografis selama 5 tahun.


In [2]:
# 2. Inspect Qdrant Vector Collections & Payload Statistics
from qdrant_client import QdrantClient
client = QdrantClient(host="127.0.0.1", port=6333)
collections = client.get_collections().collections
print(f"Total Collections: {len(collections)}")
for col in collections:
    info = client.get_collection(col.name)
    print(f" - {col.name:<25}: {info.points_count} points | Vector Dim: {info.config.params.vectors.size}")

Total Collections: 6
 - court_decisions          : 0 points | Vector Dim: 768
 - legal_commentaries       : 0 points | Vector Dim: 768
 - legal_concepts           : 0 points | Vector Dim: 768
 - legal_research           : 0 points | Vector Dim: 768
 - regulations              : 0 points | Vector Dim: 768
 - synthetic_legal_corpus   : 2 points | Vector Dim: 768


In [3]:
# 3. Test Local GPU Inference via Ollama
import ollama
prompt = "Jelaskan mengapa asas kepastian hukum dan keterlacakan (traceability) krusial dalam penerapan AI di sektor peradilan (maksimal 3 butir kalimat):"
response = ollama.generate(model="qwen2.5:3b", prompt=prompt, options={"temperature": 0.2})
print(response["response"])


Asas kepastian hukum dan keterlacakan adalah krusial dalam penerapan AI di sektor peradilan karena mereka membantu memastikan keadilan dan transparansi dalam penggunaan teknologi ini. Keterlacakan memastikan bahwa AI tidak membuat kesalahan yang signifikan dan dapat diikuti, sementara kepastian hukum memastikan bahwa keputusan AI tidak dapat digunakan untuk tujuan yang tidak sah atau tidak adil. Ini penting untuk memastikan kepercayaan publik terhadap sistem peradilan yang menggunakan AI.


In [4]:
# 4. Execute End-to-End Hybrid Search & Grounded Drafting
import sys
sys.path.append("/home/ai/JPAIS/projects/jpais_core")
from pipeline import JPAISCorePipeline

pipeline = JPAISCorePipeline()
query = "Berapa lama jangka waktu wajib penyimpanan catatan asal-usul data sistem AI pengadilan?"
print(pipeline.hybrid_search_and_draft(query))
pipeline.close()


Executing Hybrid Retrieval for: 'Berapa lama jangka waktu wajib penyimpanan catatan asal-usul data sistem AI pengadilan?'
 -> Qdrant retrieved Article: SYN-REG-2026-001-ART-12 (Status: synthetic)
 -> Neo4j resolved graph context: Regulation='Undang-Undang Tata Kelola Kecerdasan Artifisial Berdaulat', Concept='Pencatatan Asal-Usul Data (Provenance Logging)'
 -> Submitting grounded context to local Ollama (qwen2.5:3b)...
[PERINGATAN TATA KELOLA DATA JPAIS]
Status Korpus: SINTETIS (Untuk Keperluan Pengujian & Validasi Teknis).
Advis ini TIDAK DAPAT digunakan sebagai rujukan hukum positif atau otoritatif peradilan.
Wajib melalui penelaahan manusia (Human Review) oleh praktisi hukum sebelum digunakan.

Advis hukum singkat berdasarkan ketentuan yang diberikan:

"Menurut Pasal 12 ayat (1) Undang-Undang Tata Kelola Kecerdasan Artifisial Berdaulat (SYN-REG-2026-001), setiap institusi peradilan yang mengoperasikan kecerdasan buatan wajib mempertahankan log asal-usul data (provenance) secara kri